In [0]:
dbutils.widgets.text("catalog_name", "dbr_dev")
dbutils.widgets.text("container", "trezio2005")
dbutils.widgets.text("storage_account", "dlspl21databricks")
dbutils.widgets.text("volume_name", "raw_data")

catalog_name = dbutils.widgets.get("catalog_name")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")
volume_name = dbutils.widgets.get("volume_name")

volume_path = f"/Volumes/{catalog_name}/{container}_bronze/{volume_name}/"
target_table = f"{catalog_name}.trezio2005_bronze.ohlc_candles"

checkpoint_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/checkpoints/ohlc_stream"
schema_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/schemas/ohlc_stream"

Autoloader with evolution mode

In [0]:
from pyspark.sql.functions import current_timestamp, current_date

df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "json")
      .option("cloudFiles.schemaLocation", schema_path)
      .option("cloudFiles.inferColumnTypes", "true")
      .option("cloudFiles.schemaEvolutionMode", "addNewColumns")    #autoloader will automatically update schema when encountered new columns
      .option("header", "true")
      .load(volume_path)
      .selectExpr("*", "_metadata.file_name as source_filename")
      .withColumn("ingestion_timestamp", current_timestamp())
      .withColumn("load_date", current_date())
)
df = df.toDF(*[col.replace(' ', '_') for col in df.columns])

(df.writeStream
 .format("delta")
 .option("checkpointLocation", checkpoint_path)
 .option("mergeSchema", "true") #required for delta to update new columns
 .trigger(availableNow=True)
 .toTable(target_table)
)

With schemaEvolutionMode autoloader returned information: Encountered unknown fields during parsing: {"Volume":2188}, which can be fixed by an automatic retry: true 

After retry everything went as planned and new columns have been added to the schema

In [0]:
display(spark.sql(f"""
                  SELECT * FROM {catalog_name}.trezio2005_bronze.ohlc_candles
                  ORDER BY Timestamp DESC
                  LIMIT 10
                  """))

New columns have values int he volume column so ingestion went well

Rescued_data - column where damaged or corrupted data is stored

In [0]:
#Streaming Statistics of the ingestion
  "sources" : [ {
    "description" : "CloudFilesSource[/Volumes/dbr_dev/trezio2005_bronze/raw_data/]",
    "startOffset" : {
      "seqNum" : 1443,
      "sourceVersion" : 3,
      "lastBackfillStartTimeMs" : 1784727576937,
      "lastBackfillFinishTimeMs" : 1784727625123,
      "lastInputPath" : "/Volumes/dbr_dev/trezio2005_bronze/raw_data/"
    },
    "endOffset" : {
      "seqNum" : 1449,
      "sourceVersion" : 3,
      "lastBackfillStartTimeMs" : 1784727576937,
      "lastBackfillFinishTimeMs" : 1784727625123,
      "lastInputPath" : "/Volumes/dbr_dev/trezio2005_bronze/raw_data/"
    },
    "latestOffset" : null,
    "numInputRows" : 5,     #number of rows ingested - correct
    "inputRowsPerSecond" : 0.0,
    "processedRowsPerSecond" : 0.3,
    "metrics" : {
      "isBacklogComputationComplete" : "true",
      "numBytesOutstanding" : "0",
      "numFilesOutstanding" : "0"
    }
  } ]